# RNG seeding control

Seed the task stream and action sampling independently. A new task gets a new seed; internal episode resets continue the underlying RNG. Repeating both seeds reproduces the rollout.


In [ ]:
"""Reproduce complete task sequences using independent action sampling."""

import gymnasium as gym
import numpy as np

from repeat_task_gym import RepeatTaskEnv


def rollout(task_seed, action_seed):
    env = RepeatTaskEnv(
        gym.make("CartPole-v1", max_episode_steps=5), max_task_episodes=2
    )
    env.action_space.seed(action_seed)
    rows = []
    task_starts = []
    try:
        observation, _ = env.reset(seed=task_seed)
        task_starts.append(observation["observation"].copy())
        for _ in range(40):
            action = env.action_space.sample()
            observation, reward, terminated, truncated, _ = env.step(action)
            rows.append(
                (
                    observation["observation"].copy(),
                    observation["episode_done"],
                    reward,
                    terminated,
                    truncated,
                )
            )
            if terminated or truncated:
                observation, _ = env.reset()
                task_starts.append(observation["observation"].copy())
    finally:
        env.close()
    return rows, np.array(task_starts)


def main():
    first, starts = rollout(123, 456)
    replay, replay_starts = rollout(123, 456)
    for left, right in zip(first, replay, strict=True):
        np.testing.assert_array_equal(left[0], right[0])
        assert left[1:] == right[1:]
    np.testing.assert_array_equal(starts, replay_starts)
    assert not np.array_equal(starts[0], starts[1])

    _, different_starts = rollout(124, 456)
    assert not np.array_equal(starts, different_starts)
    print(f"Reproduced {len(first)} frames across {len(starts)} task starts.")


if __name__ == "__main__":
    main()
